In [ ]:
import numpy as np

In [2]:
class MatrixFactorizationSGD:
    """
    Matrix Factorization with SGD — built directly on your SGDRegressor style
    Predicts rating = user_factor[u] dot item_factor[i]
    Added two parameters: n_users & n_items to make the model more robust 
    added user_bias and item_bias
    """
    def __init__(self, n_factors=20, learning_rate=0.01, epochs=50, 
                 batch_size=1, reg=None, reg_param=0.01, n_users=None, n_items=None):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg = reg
        self.reg_param = reg_param
        self.n_users = n_users         
        self.n_items = n_items
        
        self.P = None  # user latent factors
        self.Q = None  # item latent factors
        self.global_bias = 0.0
        self.user_bias = None
        self.item_bias = None
        
    def fit(self, user_ids, item_ids, ratings):
        """
        user_ids, item_ids, ratings: numpy arrays (or pandas Series) of same length
        """
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        ratings = np.asarray(ratings, dtype=float)
        
        n_users = self.n_users if self.n_users is not None else (user_ids.max() + 1)
        n_items = self.n_items if self.n_items is not None else (item_ids.max() + 1)
        
        # Initialize latent factors (small random values)
        self.P = np.random.normal(0, 0.01, (n_users, self.n_factors))
        self.Q = np.random.normal(0, 0.01, (n_items, self.n_factors))
        self.global_bias = np.mean(ratings)

        self.user_bias = np.random.normal(0, 0.001, n_users)  # indicates the observed deviation from users
        self.item_bias = np.random.normal(0, 0.001, n_items) 
        # indicates the observed deviation from items and just to experiment, I added little noise
        
        for epoch in range(self.epochs):
            # Shuffle (stochastic part)
            indices = np.random.permutation(len(ratings))
            for i in range(0, len(ratings), self.batch_size):
                batch_index = indices[i:i + self.batch_size]
                u_batch = user_ids[batch_index]
                i_batch = item_ids[batch_index]
                r_batch = ratings[batch_index]
                
                # Forward pass
                pred = (np.sum(self.P[u_batch] * self.Q[i_batch], axis=1) + self.global_bias 
                        + self.user_bias[u_batch] + self.item_bias[i_batch])
                error = r_batch - pred
                
                # Gradients (same style as your SGDRegressor!)
                grad_P = -2 * (error[:, np.newaxis] * self.Q[i_batch]) # chain rule is applied here
                grad_Q = -2 * (error[:, np.newaxis] * self.P[u_batch])

                grad_u_bias = -2 * error
                grad_i_bias = -2 * error
                
                # Regularization (exactly like your code)
                if self.reg == 'l2':
                    grad_P += 2 * self.reg_param * self.P[u_batch]
                    grad_Q += 2 * self.reg_param * self.Q[i_batch]
                    grad_u_bias += 2 * self.reg_param * self.user_bias[u_batch]
                    grad_i_bias += 2 * self.reg_param * self.item_bias[i_batch]
                    
                elif self.reg == 'l1':
                    grad_P += self.reg_param * np.sign(self.P[u_batch])
                    grad_Q += self.reg_param * np.sign(self.Q[i_batch])
                    grad_u_bias += self.reg_param * np.sign(self.user_bias[u_batch])
                    grad_i_bias += self.reg_param * np.sign(self.item_bias[i_batch])
                
                # SGD update
                self.P[u_batch] -= self.learning_rate * grad_P
                self.Q[i_batch] -= self.learning_rate * grad_Q
                self.user_bias[u_batch] -= self.learning_rate * grad_u_bias  
                self.item_bias[i_batch] -= self.learning_rate * grad_i_bias
        
        return self
    
    def predict(self, user_ids, item_ids):
        user_ids = np.asarray(user_ids, dtype=int)
        item_ids = np.asarray(item_ids, dtype=int)
        
        return (np.sum(self.P[user_ids] * self.Q[item_ids], axis=1) 
                + self.global_bias 
                + self.user_bias[user_ids] 
                + self.item_bias[item_ids])
    